# MechanicsKit `mk.md` Examples

`mk.md(template)` is a markdown helper that interpolates variables from the calling cell's namespace and dispatches on type, producing **markdown source** (not HTML). Same value renders correctly in marimo, Jupyter, and quarto.

## What it does over a plain f-string

- No `f` prefix — `{var}` looks up `var` in the cell scope.
- Type-aware rendering: ndarrays become markdown tables (or `bmatrix` when large), DataFrames become markdown tables, `LatexArray` / `ltx` results pass through their LaTeX, scalars get standard format specs.
- Returns a `Markdown` object whose primary representation is markdown source — Jupyter and quarto render it through their own pipelines.

## Brace escaping

`mk.md` uses Python's `str.format` rules for `{` / `}`. To write a literal brace (e.g. for LaTeX), double it: `\dfrac{{1}}{{2}}` renders as `\dfrac{1}{2}`.

In [ ]:
import numpy as np
import mechanicskit as mk
from mechanicskit import md, la, ltx

## Example 1: Scalars and format specs

Plain variable lookup and standard Python format specs (`:.2f`, `:>10`, etc.) work identically to f-strings.

In [ ]:
a = 3.14159
md("$a = {a:.2f}$")

## Example 2: Arithmetic inside the placeholder

Unlike plain `str.format`, `mk.md` evaluates the placeholder text as a Python expression, so arithmetic, attribute access, and method calls all work.

In [ ]:
F_st = 89000.0
md("Pre-tension force: $F_0 = {F_st/1000:.2f}$ kN")

## Example 3: NumPy 2-D array → markdown table

Small (≤8×8) 2-D arrays render as a markdown table — no manual formatting, no `repr(ndarray)` falling into the output.

In [ ]:
K = np.array([[2, -1, 0],
              [-1, 2, -1],
              [0, -1, 2]])
eigs = np.linalg.eigvalsh(K).round(3)
md("""
The element stiffness matrix is

{K}

with eigenvalues {eigs}.
""")

## Example 4: Large matrix → truncated `bmatrix`

Matrices larger than 8×8 fall back to a truncated LaTeX `bmatrix` (matching `LatexArray`'s 8×8 truncation).

In [ ]:
B = np.arange(100).reshape(10, 10).astype(float)
md("$B = {B}$")

## Example 5: LaTeX composition with `ltx`

When you want LaTeX rendering of a matrix instead of a markdown table, compose with `ltx` and interpolate the result. The `Markdown` wrapper's `__format__` returns markdown source, so nesting just works.

In [ ]:
K = np.array([[2, -1], [-1, 2]])
md("Stiffness: {K_ltx}", K_ltx=ltx(r"K = ", K, r"\,N/m"))

## Example 6: Pandas DataFrame → markdown table

`DataFrame.to_markdown()` (which uses `tabulate` internally) is wired in automatically.

In [ ]:
import pandas as pd
df = pd.DataFrame({
    "node": [1, 2, 3, 4],
    "u_x":  [0.0, 1.2, 2.4, 3.1],
    "u_y":  [0.0, 0.5, 0.8, 0.4],
})
md("""
## Nodal displacements

{df}

Total: {df.shape[0]} nodes.
""")

## Example 7: list-of-dicts → markdown table

Useful for ad-hoc tables built from loops, without reaching for pandas.

In [ ]:
elements = [
    {"id": 1, "type": "ROD",   "length": 1.5, "E": 200e9},
    {"id": 2, "type": "ROD",   "length": 2.0, "E": 200e9},
    {"id": 3, "type": "BEAM",  "length": 1.0, "E": 70e9},
]
md("### Element table\n\n{elements}")

## Example 8: Real-world parameter table

Mixing inline LaTeX, attribute access, arithmetic, and format specs in one block — this is the kind of cell `mk.md` is built for. Note how `$F_{{st}}$` writes a literal `$F_{st}$` (doubled braces because `mk.md` uses `str.format`).

In [ ]:
from types import SimpleNamespace
skruvStandard = SimpleNamespace(value=12)
P     = 1.75       # mm  pitch
As    = 84.3       # mm² stress area
Db    = 10.2       # mm  drill diameter
dh    = 13.0       # mm  clearance hole
s     = 19.0       # mm  head width
F_st  = 89_000     # N   yield force
F_0   = 0.73 * F_st
M_at  = 84.5       # Nm  tightening torque
M_loss = 78.2      # Nm  loosening torque

md(r"""
| Beteckning   | Beskrivning                             | Värde                  | Enhet      |
|:---          |:---                                     |:---                    |:---        |
| $d$          | Skruvgäng metrisk diameter              | M{skruvStandard.value} | M standard |
| $P$          | Gängstigning grov                       | {P}                    | mm         |
| $A_s$        | Spänningsarea                           | {As}                   | mm$^2$     |
| $D_b$        | Borrdiameter före gängning              | {Db}                   | mm         |
| $d_h$        | Frigående håldiameter                   | {dh}                   | mm         |
| $d_w$        | Skruvhuvud storlek                      | {s}                    | mm         |
| $F_{{st}}$   | Sträckkraft $R_{{p02}} \cdot A_s$       | {F_st/1000:0.2f}       | kN         |
| $F_0$        | Förspänningskraft $0.73 \cdot F_{{st}}$ | {F_0/1000:0.2f}        | kN         |
| $M_{{at}}$   | Åtdragningsmoment                       | {M_at:0.2f}            | Nm         |
| $M_{{loss}}$ | Lossningsmoment                         | {M_loss:0.2f}          | Nm         |
""")

## Example 9: Raw markdown source

Sometimes you just want the resulting markdown text — for writing out to a `.md` / `.qmd` file, debugging, or piping into another tool. `str()` on the result gives you the source verbatim.

In [ ]:
result = md("## $F = {F_st/1000:.2f}$ kN")
print(str(result))